[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/03_class_weighting_vs_thresholds.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Class weighting versus moving the threshold

Allow about 30 minutes. Does a weighted training loss buy a better review
policy? You will fit two logistic regressions, separate a change in the
probability scale from a change in ranking, and compare the priced losses each
can reach. The two features are fully synthetic: no ECG was measured.

The action is **send this recording for AF review**, priced as in the core
notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0, in illustrative loss units.
Replace each `...` and run the checker below it; if you are stuck for about
10 minutes, use the stuck switch below the function, where there is one, or open the worked copy in `extension/solutions/`.
A later cell that needs an unfinished problem prints `Waiting for ...`: once that
problem is done or switched, click in the waiting cell, choose **Runtime > Run
before**, then run the waiting cell.

In [ ]:
import numpy as np
from IPython.display import display
from clinical_utility import reference, calibration, comparison, logistic, cost_sensitive
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
dataset = cost_sensitive.synthetic_rhythm_features()
train, test = dataset["split"] == "train", dataset["split"] == "test"
X_train, y_train = dataset["X"][train], dataset["y"][train]
X_test, y_test = dataset["X"][test], dataset["y"][test]

## What weighting asks the fit to learn

AF is uncommon, so a common recipe gives each AF training example a larger
weight `w` than each example without AF. The two features imitate RR
irregularity and P-wave absence. Within each class they are Gaussian with a
shared covariance, so the true log-odds of AF are linear in the features and
an unweighted logistic regression is correctly specified.

### Problem A: Predict

Which part of the fitted model should weighting mostly change here: the
`"intercept"` or the `"ranking"` of recordings? Record your prediction before
reading on.

In [ ]:
weight_prediction = "intercept"
_ = check.run("weight_predict", weight_prediction)

The argument uses the log-odds scale. The logit of a probability `p` is its
log-odds, `logit(p) = log(p / (1 - p))`, and the sigmoid,
`sigmoid(z) = 1 / (1 + exp(-z))`, turns log-odds back into a probability.

Let `p(x)` be the ordinary AF probability at feature value `x`. With weight `w`
on AF, the weighted objective at `x` is proportional to
`w*p*log(q) + (1-p)*log(1-q)`. It is maximised at `q = w*p / (w*p + 1 - p)`,
that is

`logit(q) = logit(p) + log(w)`.

When the ordinary log-odds are linear in the features, only the intercept
changes in the population optimum; in a finite sample the slopes move a little
too. Multiplying every weight by the same constant changes nothing.

For example, a weight ratio of 9 turns an ordinary probability of 0.1 into a
weighted output of 0.5: logit(0.1) = -2.20, adding
log 9 = 2.20 gives 0, and sigmoid(0) = 0.5. The recording and its features are the same; the fit
now answers a reweighted question. Reviewing above 0.5 on the weighted scale is
therefore a different cutoff on the ordinary scale, and the weighted output is
not a calibrated probability for the original population.

## Fit both objectives

### Problem B: Make

Implement `balanced_weights(y)`: return an array with weight `n/(2*n_class)`
for each training row, in its original order, so that both classes carry the
same total weight. Do not compute weights from the test labels.

In [ ]:
def balanced_weights(y):
    y = np.asarray(y, dtype=int)
    counts = np.bincount(y, minlength=2)
    return len(y) / (2 * counts[y])
_ = check.run("weight_weights", balanced_weights)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# balanced_weights = check.use_reference("balanced_weights")

### Problem C: Modify

`fit_weighted` below is a working Newton fit for logistic regression, with an
intercept column and a step that halves until the weighted log-likelihood
improves. As written it ignores `row_weights`. Change the gradient and
information lines so that each row counts with its weight:

- gradient: `X.T @ (w * (y - p))`;
- information: `X.T @ ((w * p * (1 - p))[:, None] * X)`.

The weights must enter the gradient, which defines where the optimum lies. Put
them in the information matrix too, so each step is a true Newton step.

In [ ]:
def fit_weighted(features, y, row_weights):
    """Logistic regression coefficients, intercept first, with one weight per row."""
    X = np.column_stack([np.ones(len(y)), features])
    beta = np.zeros(X.shape[1])

    def log_likelihood(b):
        z = X @ b
        return row_weights @ (y * z - np.logaddexp(0, z))

    for _ in range(100):
        p = logistic.sigmoid(X @ beta)
        gradient = X.T @ (row_weights * (y - p))
        information = X.T @ ((row_weights * p * (1 - p))[:, None] * X)
        step = np.linalg.solve(information, gradient)
        scale = 1.
        while log_likelihood(beta + scale * step) < log_likelihood(beta) - 1e-12:
            scale /= 2
        beta = beta + scale * step
        if np.max(np.abs(scale * step)) < 1e-10:
            break
    return beta


_ = check.run("weight_fit", fit_weighted)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# fit_weighted = check.use_reference("fit_weighted")

In [ ]:
def fit_both():
    """Coefficients and test-set probabilities, fitted without and with balanced weights."""
    fits = {"unweighted": fit_weighted(X_train, y_train, np.ones(len(y_train))),
            "weighted": fit_weighted(X_train, y_train, balanced_weights(y_train))}
    probabilities = {name: logistic.sigmoid(beta[0] + X_test @ beta[1:]) for name, beta in fits.items()}
    return fits, probabilities


# Later cells call fit_both() again, so each also runs on its own once B and C pass.
if check.ready(balanced_weights=balanced_weights, fit_weighted=fit_weighted):
    fits, probabilities = fit_both()
    display(view.coefficients(fits, dataset["feature_names"]))
    a, b = fits["unweighted"], fits["weighted"]
    print("Cosine between the two slope vectors:",
          a[1:] @ b[1:] / (np.linalg.norm(a[1:]) * np.linalg.norm(b[1:])))
    print("Intercept shift:", b[0] - a[0])
    for name, p in probabilities.items():
        counts = comparison.counts_at_threshold(p, y_test, .5)
        print(name, "AUC", comparison.auc(p, y_test), "reviews at 0.5", counts["TP"] + counts["FP"],
              "loss at 0.5", reference.matrix_loss(counts, costs))

The cosine between the two slope vectors is 0.997, and the test AUCs are
0.909 unweighted and 0.907 weighted. The intercept moves by 2.27, close to the
log weight ratio 2.20. At a cutoff of 0.5 the reviews change from 49 to
233. More reviews at a fixed cutoff do not show that weighting learned a
better discriminator.

## Let both fits choose a threshold

### Problem D: Make

Write `minimum_loss(scores, y, costs)`, returning one number. Try every distinct
score as a cutoff,
plus review-all and review-none. Count TP, FP, FN and TN with boolean masks,
charge all four prices and return the lowest total loss, not the cutoff. The
checker has tied scores and charges correct reviews, so leaving out $C_{TP}$
gives the wrong answer.

This is a **hindsight comparison** on the test set: what could each ranking
achieve? It is optimistic about any policy you would actually select. For a
deployment claim, choose the cutoff on development data and assess the frozen
policy on untouched data.

In [ ]:
def minimum_loss(scores, y, costs):
    scores, y = np.asarray(scores), np.asarray(y, dtype=bool)
    losses = []
    for cut in np.r_[0., np.unique(scores), 1.01]:
        review = scores >= cut
        counts = {"TP": np.sum(review & y), "FP": np.sum(review & ~y),
                  "FN": np.sum(~review & y), "TN": np.sum(~review & ~y)}
        losses.append(sum(counts[key] * costs[key] for key in counts))
    return float(min(losses))
_ = check.run("weight_optimum", minimum_loss)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# minimum_loss = check.use_reference("minimum_loss")

In [ ]:
if check.ready(balanced_weights=balanced_weights, fit_weighted=fit_weighted,
               minimum_loss=minimum_loss):
    fits, probabilities = fit_both()
    print({name: minimum_loss(p, y_test, costs) for name, p in probabilities.items()})

The exact minimum losses are 299 and 303. Moving the threshold recovers
almost the same priced policy from either fit; the small slope changes in a
finite sample explain why the two minima need not be identical.

## Recover the probability scale

### Problem E: Fix the bug

A report treats the weighted model's outputs as AF probabilities for the
original population, after the correction below. Fix `undo_weighting` in place
using the derivation above. This is an idealised correction; it does not
replace checking calibration on held-out recordings.

In [ ]:
def undo_weighting(q, positive_to_negative_weight):
    """Original-population probability from a weighted model's output."""
    return logistic.sigmoid(logistic.logit(q) - np.log(positive_to_negative_weight))


_ = check.run("weight_correct", undo_weighting)

In [ ]:
if check.ready(balanced_weights=balanced_weights, fit_weighted=fit_weighted,
               undo_weighting=undo_weighting):
    fits, probabilities = fit_both()
    train_weights = balanced_weights(y_train)
    ratio = train_weights[y_train == 1][0] / train_weights[y_train == 0][0]
    corrected = undo_weighting(probabilities["weighted"], ratio)
    print("Test Brier, weighted and corrected:",
          calibration.brier_score(probabilities["weighted"], y_test),
          calibration.brier_score(corrected, y_test))

## Before you finish

- Distinguish training weights, output probabilities and the review threshold.
- Recover the log weight ratio in the intercept-shift derivation.
- Compare priced minima rather than review counts at one cutoff.
- Explain why a hindsight optimum needs fresh evaluation before deployment.

**Discuss:** With misspecified log-odds, or a flexible model of limited
capacity, how might weighting change which patterns are learned? What
experiment would tell that apart from an intercept shift?